# Camada Silver — ecommerce_pedidos

## Objetivo

Aplicar regras de Data Quality sobre os dados provenientes da camada Bronze da tabela `ecommerce_pedidos`.

O processamento deve permitir:

- identificar registros válidos;
- identificar registros com falhas de qualidade;
- registrar os motivos das falhas por meio de flags booleanas;
- preparar os dados para persistência na camada Silver.

### Fluxo

Bronze → Regras de Data Quality → Flags de qualidade → Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from deltalake import DeltaTable, write_deltalake

In [0]:
%run ./00_autenticação

In [0]:
storage_account = "internshipdatalake"
lake_container = "squad1"

bronze_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

silver_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"silver/ecommerce_pedidos"
)

storage_options = {
    "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
    "AZURE_STORAGE_CLIENT_ID": client_id,
    "AZURE_STORAGE_TENANT_ID": tenant_id,
    "AZURE_STORAGE_CLIENT_SECRET": client_secret
}

In [0]:
# Leitura da camada Bronze

dt_bronze = DeltaTable(
    bronze_path,
    storage_options=storage_options
)

tabela_bronze = (
    dt_bronze
    .to_pyarrow_table()
)

df_bronze = spark.createDataFrame(
    tabela_bronze.to_pandas()
)

print(
    "Registros recebidos da Bronze:",
    df_bronze.count()
)

df_bronze.printSchema()

In [0]:
# Define os valores permitidos para campos com domínio fechado.

status_validos = [
    "Processando",
    "Pagamento Aprovado",
    "Em Separação",
    "Enviado",
    "Entregue",
    "Cancelado"
]

metodos_pagamento_validos = [
    "cartão crédito",
    "cartão débito",
    "pix",
    "boleto"
]

In [0]:
print("=== STATUS_PEDIDO ===")

df_bronze.groupBy(
    "status_pedido"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)

print("=== METODO_PAGAMENTO ===")

df_bronze.groupBy(
    "metodo_pagamento"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)

In [0]:
# Aplicação das regras de Data Quality
# Convenção:
# True  = registro falhou na regra
# False = registro passou na regra

df_dq = (
    df_bronze

    # Regra 1
    # id_pedido não pode ser nulo ou duplicado
    # PENDENTE: alinhar granularidade da Silver e tratamento
    # de múltiplas versões do mesmo pedido.

    # Regra 2
    # id_cliente não pode ser nulo.
    .withColumn(
        "dq_id_cliente_nulo",
        F.col("id_cliente").isNull()
    )

    # Regra 3
    # status_pedido deve pertencer ao domínio permitido.
    .withColumn(
        "dq_status_invalido",
        F.col("status_pedido").isNull()
        | ~F.col("status_pedido").isin(status_validos)
    )

    # Regra 4
    # valor_total deve ser maior que zero.
    .withColumn(
        "dq_valor_total_invalido",
        F.col("valor_total").isNull()
        | (F.col("valor_total") <= 0)
    )

    # Regra 5
    # metodo_pagamento deve pertencer ao domínio permitido.
    .withColumn(
        "dq_metodo_pagamento_invalido",
        F.col("metodo_pagamento").isNull()
        | ~F.col("metodo_pagamento").isin(
            metodos_pagamento_validos
        )
    )

    # Regra 6
    # dt_ultima_atualizacao_status não pode ser anterior a dt_pedido.
    .withColumn(
        "dq_data_status_invalida",
        F.col("dt_pedido").isNotNull()
        & F.col("dt_ultima_atualizacao_status").isNotNull()
        & (
            F.col("dt_ultima_atualizacao_status")
            < F.col("dt_pedido")
        )
    )

    # Regra 7
    # Pedidos acima de R$ 300 devem possuir frete igual a zero.
    .withColumn(
        "dq_frete_invalido",
        (F.col("valor_total") > 300)
        & (
            F.col("valor_frete").isNull()
            | (F.col("valor_frete") != 0)
        )
    )

    # Regra 8
    # Pedidos cancelados não devem possuir rastreamento ativo.
    # PENDENTE: depende da tabela ecommerce_rastreamento_entregas.

    # Regra 9
    # Pedidos entregues devem possuir atualização de status
    # pelo menos 2 dias após a data do pedido.
    .withColumn(
        "dq_entrega_prazo_invalido",
        F.when(
            F.col("status_pedido") == "Entregue",
            F.col("dt_pedido").isNull()
            | F.col("dt_ultima_atualizacao_status").isNull()
            | (
                F.col("dt_ultima_atualizacao_status")
                < (
                    F.col("dt_pedido")
                    + F.expr("INTERVAL 2 DAYS")
                )
            )
        ).otherwise(False)
    )

    # Regra 10
    # id_endereco_entrega deve existir em ecommerce_enderecos.
    # PENDENTE: leitura da tabela de endereços ainda bloqueada
    # no ambiente atual.
)

In [0]:
# Resumo das falhas identificadas pelas regras implementadas

df_dq.select(
    F.sum(
        F.col("dq_id_cliente_nulo").cast("int")
    ).alias("regra_02_id_cliente"),

    F.sum(
        F.col("dq_status_invalido").cast("int")
    ).alias("regra_03_status"),

    F.sum(
        F.col("dq_valor_total_invalido").cast("int")
    ).alias("regra_04_valor_total"),

    F.sum(
        F.col("dq_metodo_pagamento_invalido").cast("int")
    ).alias("regra_05_pagamento"),

    F.sum(
        F.col("dq_data_status_invalida").cast("int")
    ).alias("regra_06_datas"),

    F.sum(
        F.col("dq_frete_invalido").cast("int")
    ).alias("regra_07_frete"),

    F.sum(
        F.col("dq_entrega_prazo_invalido").cast("int")
    ).alias("regra_09_entrega")
).show()

------------------------------------


## Dependências pendentes

As regras abaixo ainda dependem de definição ou de fontes externas antes da persistência da camada Silver:

- **Regra 1 — `id_pedido` nulo ou duplicado:** pendente de alinhamento sobre a granularidade da Silver e o tratamento de múltiplas versões do mesmo pedido.
- **Regra 8 — pedidos cancelados sem rastreamento ativo:** depende da tabela `ecommerce_rastreamento_entregas`, ainda não disponível/localizada na Bronze da Squad 1.
- **Regra 10 — endereço de entrega válido:** depende da tabela `ecommerce_enderecos`. A tabela existe fisicamente na Bronze, porém sua leitura está bloqueada no ambiente atual por incompatibilidade com Deletion Vectors no `deltalake` e pela configuração de acesso ABFSS via Spark.

> A persistência definitiva da Silver deve ocorrer somente após a resolução dessas dependências.